In [ ]:
# In[1]:

import pandas as pd
import pytz
from pathlib import Path

# Load CSVs into kernel variables (reuse variable names if rerun)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error_logs = pd.read_csv("error_logs.csv")

# Helper to safely summarize a dataframe file
def summarize_df(df, file_label, kpi_col=None, max_cmdb=50, max_kpi=50, max_msgs=10):
    # Row count
    rows = int(df.shape[0])
    # Timestamps: if 'timestamp' column exists and non-empty
    if 'timestamp' in df.columns and not df['timestamp'].dropna().empty:
        min_ts = int(df['timestamp'].min())
        max_ts = int(df['timestamp'].max())
    else:
        min_ts = None
        max_ts = None
    # cmdb_id samples
    if 'cmdb_id' in df.columns:
        uniq_cmdb = df['cmdb_id'].dropna().astype(str).unique()
        sample_cmdb = list(uniq_cmdb[:max_cmdb])
        cmdb_count = int(len(uniq_cmdb))
    else:
        sample_cmdb = []
        cmdb_count = 0
    # KPI/feature names or message previews
    sample_kpis = []
    kpi_count = 0
    message_examples = []
    message_exists = False
    if file_label == 'metric':
        if kpi_col and kpi_col in df.columns:
            uniq_kpis = df[kpi_col].dropna().astype(str).unique()
            sample_kpis = list(uniq_kpis[:max_kpi])
            kpi_count = int(len(uniq_kpis))
    elif file_label == 'trace':
        if kpi_col and kpi_col in df.columns:
            uniq_kpis = df[kpi_col].dropna().astype(str).unique()
            sample_kpis = list(uniq_kpis[:max_kpi])
            kpi_count = int(len(uniq_kpis))
    elif file_label == 'log':
        if kpi_col and kpi_col in df.columns:
            uniq_kpis = df[kpi_col].dropna().astype(str).unique()
            sample_kpis = list(uniq_kpis[:max_kpi])
            kpi_count = int(len(uniq_kpis))
    elif file_label == 'error_logs':
        message_exists = 'message' in df.columns
        if message_exists:
            # Take distinct example messages, truncated to 200 chars to keep compact
            uniq_msgs = df['message'].dropna().astype(str).map(lambda s: s[:200]).unique()
            message_examples = list(uniq_msgs[:max_msgs])
    # Build compact summary dict
    return {
        'file': file_label,
        'rows': rows,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'cmdb_unique_count': cmdb_count,
        'cmdb_sample_up_to_50': sample_cmdb,
        'kpi_unique_count': kpi_count,
        'kpi_sample_up_to_50': sample_kpis,
        'message_exists': message_exists,
        'message_examples_up_to_10': message_examples
    }

# Summaries for each file
sum_metric = summarize_df(df_metric, 'metric', kpi_col='kpi_name')
sum_trace = summarize_df(df_trace, 'trace', kpi_col='trace_name')
sum_log = summarize_df(df_log, 'log', kpi_col='log_name')
sum_error = summarize_df(df_error_logs, 'error_logs')

# Create a compact DataFrame summary (one row per file)
summary_df = pd.DataFrame([sum_metric, sum_trace, sum_log, sum_error])[
    ['file','rows','min_timestamp','max_timestamp',
     'cmdb_unique_count','cmdb_sample_up_to_50',
     'kpi_unique_count','kpi_sample_up_to_50',
     'message_exists','message_examples_up_to_10']
]

# Display the compact summary (kernel variable)
summary_df

```
Out[1]:
```
# Plain English summary as a single string (compact)
summary = (
    "Telemetry files loaded (timestamps shown as epoch seconds, system timezone UTC+8):\n\n"
    "1) metric.csv\n"
    f"   - rows: {sum_metric['rows']}\n"
    f"   - min_timestamp: {sum_metric['min_timestamp']}, max_timestamp: {sum_metric['max_timestamp']}\n"
    f"   - unique cmdb_id count: {sum_metric['cmdb_unique_count']}; sample (up to 10): {sum_metric['cmdb_sample_up_to_50'][:10]}\n"
    f"   - unique kpi_name count: {sum_metric['kpi_unique_count']}; sample kpis (up to 10): {sum_metric['kpi_sample_up_to_50'][:10]}\n\n"
    "2) trace.csv\n"
    f"   - rows: {sum_trace['rows']}\n"
    f"   - min_timestamp: {sum_trace['min_timestamp']}, max_timestamp: {sum_trace['max_timestamp']}\n"
    f"   - unique cmdb_id count: {sum_trace['cmdb_unique_count']}; sample (up to 10): {sum_trace['cmdb_sample_up_to_50'][:10]}\n"
    f"   - unique trace_name count: {sum_trace['kpi_unique_count']}; sample trace_names (up to 10): {sum_trace['kpi_sample_up_to_50'][:10]}\n\n"
    "3) log.csv\n"
    f"   - rows: {sum_log['rows']}\n"
    f"   - min_timestamp: {sum_log['min_timestamp']}, max_timestamp: {sum_log['max_timestamp']}\n"
    f"   - unique cmdb_id count: {sum_log['cmdb_unique_count']}; sample (up to 10): {sum_log['cmdb_sample_up_to_50'][:10]}\n"
    f"   - unique log_name count: {sum_log['kpi_unique_count']}; log_names: {sum_log['kpi_sample_up_to_50']}\n\n"
    "4) error_logs.csv\n"
    f"   - rows: {sum_error['rows']}\n"
    f"   - min_timestamp: {sum_error['min_timestamp']}, max_timestamp: {sum_error['max_timestamp']}\n"
    f"   - unique cmdb_id count: {sum_error['cmdb_unique_count']}; sample: {sum_error['cmdb_sample_up_to_50']}\n"
    f"   - 'message' column present: {sum_error['message_exists']}; example messages (up to 3): {sum_error['message_examples_up_to_10'][:3]}\n\n"
    "Notes: counts and samples are taken from the loaded DataFrames. Full tables were not displayed to keep the summary compact."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

file    rows  min_timestamp  max_timestamp  cmdb_unique_count                               cmdb_sample_up_to_50  kpi_unique_count                                kpi_sample_up_to_50  message_exists                          message_examples_up_to_10
0      metric  152352     1647867600     1647869340                 59  [adservice, adservice-0, adservice-1, adservic...              1292  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...           False                                                 []
1       trace   34440     1647867600     1647869340                 40  [adservice-0, adservice-1, adservice-2, adserv...               200  [trace.from_frontend-0.duration_mean, trace.fr...           False                                                 []
2         log    1800     1647867600     1647869340                 31  [adservice-0, adservice-1, adservice-2, cartse...                 2                   [log.error_count, log.row_count]           False                                                 []
3  error_logs    1115     1647867601     1647869398                  3               [frontend-2, frontend-1, frontend-0]                 0                                                 []            True  [severity: warning, message: failed to retriev...```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse df_metric already loaded in the kernel.

# 1) Compute global P95 (before any filtering) for every (cmdb_id, kpi_name)
df_thresholds = (
    df_metric.dropna(subset=['value'])  # ensure numeric values
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# 2) Filter df_metric to incident window March 21, 2022 21:00:00 to 2022-03-21 21:30:00 (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2022, 3, 21, 21, 0, 0))
end_dt = tz.localize(datetime(2022, 3, 21, 21, 30, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

df_metric_incident = df_metric[
    (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
].copy()

# 3) Merge global thresholds into incident rows so we can mark anomalies
df_inc_with_threshold = df_metric_incident.merge(
    df_thresholds, on=['cmdb_id', 'kpi_name'], how='left'
)

# Mark anomalies where value > global_P95 (if threshold missing, treat as False)
df_inc_with_threshold['is_anomaly'] = (
    (df_inc_with_threshold['global_P95'].notna()) &
    (df_inc_with_threshold['value'] > df_inc_with_threshold['global_P95'])
)

# 3) For each (cmdb_id, kpi_name) within incident window compute required metrics
def agg_group(g):
    total_points = int(g.shape[0])
    max_value = float(g['value'].max()) if total_points > 0 else np.nan
    anomaly_mask = g['is_anomaly']
    anomaly_count = int(anomaly_mask.sum())
    if anomaly_count > 0:
        earliest_anom_ts = int(g.loc[anomaly_mask, 'timestamp'].min())
    else:
        earliest_anom_ts = pd.NA
    return pd.Series({
        'total_points_in_window': total_points,
        'max_value_in_window': max_value,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anom_ts
    })

df_agg = (
    df_inc_with_threshold
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .apply(agg_group)
    .reset_index(drop=True)
)

# Merge global_P95 into aggregated results for display
df_agg = df_agg.merge(df_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Prepare sorted compact list top 20 by anomaly_count (desc)
top20_df = (
    df_agg
    .sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id', 'kpi_name', 'global_P95', 'anomaly_count', 'earliest_anomaly_timestamp',
             'max_value_in_window', 'total_points_in_window']]
    .head(20)
)

# Count how many (cmdb_id, kpi_name) pairs had at least one anomaly in the window
anomaly_pairs_count = int((df_agg['anomaly_count'] > 0).sum())

# Display compact outputs (variables)
df_thresholds.shape, df_metric_incident.shape, top20_df, anomaly_pairs_count

```
Out[2]:
```
Summary (plain English, compact):

- I computed global P95 thresholds for every (cmdb_id, kpi_name) pair and stored them in df_thresholds (shape: 5088 pairs × 3).
- I filtered df_metric to the incident window 2022-03-21 21:00:00 — 2022-03-21 21:30:00 (UTC+8) and stored it in df_metric_incident (152,352 rows × 4).
- Within that window I counted points exceeding each pair’s global P95 and aggregated per pair.

Top findings (top 20 ranked by anomaly_count, shown earlier):
- Many top pairs show anomaly_count = 2 during the 30-minute window.
- Example top pairs (cmdb_id — kpi_name — global_P95 — anomaly_count — earliest_anomaly_timestamp — max_value_in_window):
  1. adservice — runtime.java_lang_OperatingSystem_ProcessCpuTime — ~4.2047e+11 — 2 — 1647869280 — 4.20724e+11
  2. adservice2 — runtime.java_lang_OperatingSystem_ProcessCpuTime — ~2.1865e+11 — 2 — 1647869280 — 2.18940e+11
  3. adservice2 — runtime.java_lang_Threading_CurrentThreadCpuTime — ~1.6105e+10 — 2 — 1647869280 — 1.61222e+10
  4. adservice2 — runtime.java_lang_Threading_CurrentThreadUserTime — ~1.4982e+10 — 2 — 1647869280 — 1.49983e+10
  5. adservice2 — runtime.java_lang_Threading_CurrentThreadAllocatedBytes — ~1.1345e+10 — 2 — 1647869280 — 1.13655e+10
  6. node-4 — system.fs.inodes.total — ~6.4538e+09 — 2 — 1647867600 — 6.45426e+09
  (the full top-20 table was returned as a compact dataframe-like output)

- Count of (cmdb_id, kpi_name) pairs with at least one anomaly in the window: 1669.

Notes:
- global_P95 was computed across the entire df_metric series for each pair (before filtering).
- The incident window filtering used UTC+8 as requested.
- df_thresholds and df_metric_incident are available in the kernel for follow-up analysis.

The original code execution output of IPython Kernel is also provided below for reference:

((5088, 3), (152352, 4),          cmdb_id                                           kpi_name    global_P95  anomaly_count earliest_anomaly_timestamp  max_value_in_window  total_points_in_window
244    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  4.204708e+11            2.0               1647869280.0         4.207242e+11                    30.0
855   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  2.186542e+11            2.0               1647869280.0         2.189400e+11                    30.0
864   adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  1.610455e+10            2.0               1647869280.0         1.612217e+10                    30.0
866   adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  1.498150e+10            2.0               1647869280.0         1.499833e+10                    30.0
863   adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...  1.134459e+10            2.0               1647869280.0         1.136553e+10                    30.0
3505      node-4                             system.fs.inodes.total  6.453816e+09            2.0               1647867600.0         6.454261e+09                    30.0
3503      node-4                              system.fs.inodes.free  6.451221e+09            2.0               1647867600.0         6.451666e+09                    30.0
253    adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  6.150954e+09            2.0               1647869280.0         6.171112e+09                    30.0
255    adservice  runtime.java_lang_Threading_CurrentThreadUserTime  5.918042e+09            2.0               1647869280.0         5.939167e+09                    30.0
3502      node-4                                   system.disk.used  5.807983e+09            2.0               1647867960.0         5.808013e+09                    30.0
3446      node-3                             system.fs.inodes.total  4.850473e+09            2.0               1647867600.0         4.851220e+09                    30.0
3388      node-2                             system.fs.inodes.total  4.850389e+09            2.0               1647867600.0         4.850471e+09                    30.0
3329      node-1                             system.fs.inodes.total  4.850387e+09            2.0               1647867600.0         4.850470e+09                    30.0
3444      node-3                              system.fs.inodes.free  4.848520e+09            2.0               1647867600.0         4.849267e+09                    30.0
3327      node-1                              system.fs.inodes.free  4.848441e+09            2.0               1647867600.0         4.848525e+09                    30.0
3386      node-2                              system.fs.inodes.free  4.848414e+09            2.0               1647867600.0         4.848496e+09                    30.0
252    adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  4.595443e+09            2.0               1647869280.0         4.615817e+09                    30.0
3443      node-3                                   system.disk.used  4.364943e+09            2.0               1647868140.0         4.365030e+09                    30.0
3326      node-1                                   system.disk.used  4.360969e+09            2.0               1647867960.0         4.361008e+09                    30.0
3385      node-2                                   system.disk.used  4.360571e+09            2.0               1647867960.0         4.360601e+09                    30.0, 1669)```
```

In [ ]:
# In[3]:

# Reuse existing kernel variables: df_metric_incident, df_thresholds
import numpy as np
import pandas as pd

# Merge thresholds into incident data (ensure sorted)
df_inc_with_threshold = (
    df_metric_incident
    .merge(df_thresholds, on=['cmdb_id', 'kpi_name'], how='left')
    .sort_values(['cmdb_id', 'kpi_name', 'timestamp'])
    .reset_index(drop=True)
)

# Mark anomaly points (value > global_P95), require threshold present
df_inc_with_threshold['is_anomaly'] = (
    df_inc_with_threshold['global_P95'].notna() &
    (df_inc_with_threshold['value'] > df_inc_with_threshold['global_P95'])
)

# Create a segment id that increments on non-anomaly points within each (cmdb_id,kpi_name)
# This groups consecutive anomaly True rows into the same segment id per pair.
df_inc_with_threshold['not_anom_flag'] = (~df_inc_with_threshold['is_anomaly']).astype(int)
df_inc_with_threshold['segment'] = df_inc_with_threshold.groupby(
    ['cmdb_id', 'kpi_name'], sort=False
)['not_anom_flag'].cumsum()

# Extract only anomaly rows and aggregate consecutive runs by (cmdb_id,kpi_name,segment)
df_anom_rows = df_inc_with_threshold[df_inc_with_threshold['is_anomaly']].copy()

agg_runs = (
    df_anom_rows
    .groupby(['cmdb_id', 'kpi_name', 'segment'], as_index=False)
    .agg(
        start_timestamp=('timestamp', 'min'),
        end_timestamp=('timestamp', 'max'),
        length_in_points=('timestamp', 'count'),
        max_value_in_run=('value', 'max'),
        global_P95=('global_P95', 'first')
    )
)

# Compute breach_ratio safely (avoid division by zero)
agg_runs['breach_ratio'] = np.where(
    agg_runs['global_P95'] > 0,
    (agg_runs['max_value_in_run'] - agg_runs['global_P95']) / agg_runs['global_P95'],
    np.nan
)

# Keep only runs with length_in_points >= 2
df_faults = agg_runs[agg_runs['length_in_points'] >= 2].copy()

# Ensure integer timestamps
df_faults['start_timestamp'] = df_faults['start_timestamp'].astype(int)
df_faults['end_timestamp'] = df_faults['end_timestamp'].astype(int)
df_faults['length_in_points'] = df_faults['length_in_points'].astype(int)

# Prepare compact top-20 table sorted by breach_ratio desc
top20_runs = (
    df_faults
    .sort_values(by='breach_ratio', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'start_timestamp', 'end_timestamp',
             'length_in_points', 'max_value_in_run', 'global_P95', 'breach_ratio']]
    .head(20)
    .reset_index(drop=True)
)

# Summary integers
total_number_of_runs_found = int(df_faults.shape[0])
number_of_unique_cmdb_ids_with_runs = int(df_faults['cmdb_id'].nunique())

# Return compact results (DataFrame-like top20 and two summary counts)
top20_runs, total_number_of_runs_found, number_of_unique_cmdb_ids_with_runs

```
Out[3]:
```
Summary (plain English, compact):

- What I did: Using df_metric_incident and df_thresholds, I marked points where value > global_P95, grouped consecutive True points per (cmdb_id, kpi_name) into runs, kept only runs with length >= 2, and stored the full runs table as df_faults in the kernel.

- Totals:
  - Total runs found (length >= 2): 535
  - Unique cmdb_id values with runs: 53

- Top runs by breach_ratio (highest first) — examples from the top 20:
  1. node-6 — system.io.r_s: start 1647869040, end 1647869100, length 2, max_value 1126.0, global_P95 9.05, breach_ratio ≈ 123.42
  2. node-6 — system.io.rkb_s: start 1647869040, end 1647869100, length 2, max_value 4606.0, global_P95 46.10, breach_ratio ≈ 98.91
  3. emailservice-2 — container_network_receive_MB...: start 1647868320, end 1647868380, length 2, max_value 0.5519, global_P95 ≈ 0.02987, breach_ratio ≈ 17.48
  4. node-1 — system.io.avg_q_sz: start 1647867600, end 1647867660, length 2, max_value 0.41, global_P95 0.031, breach_ratio ≈ 12.23
  5. currencyservice2-0 — container_network_receive_MB...: start 1647867960, end 1647868020, length 2, max_value 0.7013, global_P95 ≈ 0.2488, breach_ratio ≈ 1.82

- Observations: Most runs in the top-20 are short (length = 2 points) and include both node-level IO/network metrics and container-level network metrics; breach ratios range from very large (>100×) down toward ~0.6–1.7× in the top-20.

- Artifacts available: the full runs table is stored in df_faults for further inspection and the top-20 results were returned as a compact table.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  start_timestamp  end_timestamp  length_in_points  max_value_in_run     global_P95  breach_ratio
0               node-6                                      system.io.r_s       1647869040     1647869100                 2      1.126000e+03       9.050000    123.419890
1               node-6                                    system.io.rkb_s       1647869040     1647869100                 2      4.606000e+03      46.100000     98.913232
2       emailservice-2  container.node-5.container_network_receive_MB....       1647868320     1647868380                 2      5.519094e-01       0.029869     17.477721
3               node-1                                 system.io.avg_q_sz       1647867600     1647867660                 2      4.100000e-01       0.031000     12.225806
4   currencyservice2-0  container.node-6.container_network_receive_MB....       1647867960     1647868020                 2      7.012792e-01       0.248824      1.818378
5               node-1                                      system.io.w_s       1647867600     1647867660                 2      1.015000e+02      36.725000      1.763785
6               node-4                        system.net.packets_in.count       1647868320     1647868380                 2      1.335010e+03     550.282500      1.426045
7               node-4                              system.net.bytes_rcvd       1647868320     1647868380                 2      1.975759e+06  829923.552500      1.380652
8               node-4                             system.net.tcp.in_segs       1647868320     1647868380                 2      2.701540e+03    1221.933500      1.210873
9               node-6                        system.net.tcp.retrans_segs       1647867780     1647867840                 2      2.800000e-01       0.129500      1.162162
10   shippingservice-1  container.node-5.container_network_receive_MB....       1647869100     1647869160                 2      3.028808e-01       0.177879      0.702731
11    paymentservice-2  container.node-5.container_network_receive_MB....       1647868080     1647868140                 2      2.871928e-01       0.169070      0.698657
12      emailservice-1  container.node-5.container_network_receive_MB....       1647868860     1647868920                 2      2.884369e-01       0.169813      0.698557
13    paymentservice-0  container.node-5.container_network_receive_MB....       1647867780     1647867840                 2      2.871947e-01       0.169445      0.694913
14   checkoutservice-2  container.node-5.container_network_receive_MB....       1647868080     1647868140                 2      3.133965e-01       0.190115      0.648458
15   checkoutservice-1  container.node-5.container_network_receive_MB....       1647867720     1647867780                 2      3.158574e-01       0.191742      0.647300
16       cartservice-2  container.node-5.container_memory_failures.hie...       1647868500     1647868560                 2      1.350000e+02      83.500000      0.616766
17       cartservice-2  container.node-5.container_memory_failures.con...       1647868500     1647868560                 2      1.350000e+02      83.500000      0.616766
18  shippingservice2-0  container.node-5.container_network_receive_MB....       1647868920     1647868980                 2      3.088012e-01       0.191672      0.611090
19       redis-cart2-0  container.node-5.container_network_receive_MB....       1647868680     1647868740                 2      3.268704e-01       0.203813      0.603773, 535, 53)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import re

# Reuse df_faults from kernel

# 1) df_component_agg grouped by cmdb_id
df_comp = df_faults.copy()
# Determine component_type
df_comp['component_type'] = np.where(df_comp['cmdb_id'].str.startswith('node-'), 'node', 'container')

# Aggregate per component
df_component_agg = (
    df_comp
    .groupby('cmdb_id', as_index=False)
    .agg(
        component_type=('component_type', 'first'),
        run_count=('cmdb_id', 'size'),
        earliest_start_timestamp=('start_timestamp', 'min'),
        max_breach_ratio=('breach_ratio', 'max'),
        unique_kpi_count=('kpi_name', 'nunique')
    )
    .sort_values(by=['run_count', 'max_breach_ratio'], ascending=[False, False])
    .reset_index(drop=True)
)

# Ensure integer timestamp column
df_component_agg['earliest_start_timestamp'] = df_component_agg['earliest_start_timestamp'].astype(int)

# Store df_component_agg in kernel (already named)
# 2) df_service_agg for non-node components
df_containers = df_comp[df_comp['component_type'] == 'container'].copy()

# Derive service_name by removing trailing -<digits> suffix; for safety lower-case
df_containers['service_name'] = df_containers['cmdb_id'].str.replace(r'-\d+$', '', regex=True)

# Aggregate per service_name
df_service_agg = (
    df_containers
    .groupby('service_name', as_index=False)
    .agg(
        service_run_count=('cmdb_id', 'size'),
        earliest_start_timestamp=('start_timestamp', 'min'),
        max_breach_ratio=('breach_ratio', 'max'),
        unique_cmdb_ids=('cmdb_id', 'nunique')
    )
    .sort_values(by='service_run_count', ascending=False)
    .reset_index(drop=True)
)

df_service_agg['earliest_start_timestamp'] = df_service_agg['earliest_start_timestamp'].astype(int)

# 3) Summary integers
total_runs_by_node = int(df_component_agg.loc[df_component_agg['component_type']=='node', 'run_count'].sum())
total_runs_by_container = int(df_component_agg.loc[df_component_agg['component_type']=='container', 'run_count'].sum())
number_of_nodes_with_runs = int(df_component_agg.loc[df_component_agg['component_type']=='node', 'cmdb_id'].nunique())
number_of_containers_with_runs = int(df_component_agg.loc[df_component_agg['component_type']=='container', 'cmdb_id'].nunique())

# 4) Earliest run overall
if df_faults.shape[0] > 0:
    earliest_row = df_faults.nsmallest(1, 'start_timestamp').loc[:, ['cmdb_id', 'kpi_name', 'start_timestamp', 'breach_ratio']].reset_index(drop=True)
    # ensure integer timestamp
    earliest_row['start_timestamp'] = earliest_row['start_timestamp'].astype(int)
else:
    earliest_row = pd.DataFrame(columns=['cmdb_id','kpi_name','start_timestamp','breach_ratio'])

# Prepare compact top-20 views
df_component_agg_top20 = df_component_agg.loc[:, ['cmdb_id','component_type','run_count','earliest_start_timestamp','max_breach_ratio','unique_kpi_count']].head(20)
df_service_agg_top20 = df_service_agg.loc[:, ['service_name','service_run_count','earliest_start_timestamp','max_breach_ratio','unique_cmdb_ids']].head(20)

# Display compact outputs (variables)
df_component_agg_top20, df_service_agg_top20, total_runs_by_node, total_runs_by_container, number_of_nodes_with_runs, number_of_containers_with_runs, earliest_row

```
Out[4]:
```
# Plain English summary as a single string (compact)
summary = (
    "Aggregates stored in kernel as df_component_agg and df_service_agg.\n\n"
    "1) Component-level (top 20 by run_count):\n"
    "- frontend2-0: 43 runs (earliest 1647867600), max_breach_ratio 0.2319\n"
    "- recommendationservice2-0: 24 runs, max_breach_ratio 0.2877\n"
    "- node-6: 23 runs (earliest 1647867600), max_breach_ratio 123.4199 (largest by far)\n"
    "- productcatalogservice2-0: 22 runs\n"
    "- frontend-1: 22 runs\n"
    "- adservice2: 22 runs\n"
    "- several productcatalogservice pods and adservice pods also in top ranks.\n\n"
    "2) Service-level (containers aggregated, top 20):\n"
    "- productcatalogservice: 59 total runs across pods (earliest 1647867600), max_breach_ratio 0.2903\n"
    "- frontend: 48 runs, max_breach_ratio 0.4695\n"
    "- recommendationservice: 48 runs, max_breach_ratio 0.4637\n"
    "- frontend2: 43 runs\n"
    "- adservice2: 38 runs\n"
    "- cartservice: 37 runs\n"
    "- emailservice shows a notably large per-run breach (max_breach_ratio ~17.48) though fewer runs.\n\n"
    "3) Summary counts:\n"
    f"- total_runs_by_node: {69}\n"
    f"- total_runs_by_container: {466}\n"
    f"- number_of_nodes_with_runs: {6}\n"
    f"- number_of_containers_with_runs: {47}\n\n"
    "4) Earliest run overall:\n"
    "- cmdb_id: adservice\n"
    "- kpi_name: app.grpc.count\n"
    "- start_timestamp: 1647867600\n"
    "- breach_ratio: 0.006036\n\n"
    "Notes: node-6 stands out with extremely high breach_ratio (123×) on IO read metric; most other high run counts are container pods for productcatalogservice, frontend, recommendationservice and adservice families. Full aggregate tables are available in df_component_agg and df_service_agg for follow-up."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id component_type  run_count  earliest_start_timestamp  max_breach_ratio  unique_kpi_count
0                frontend2-0      container         43                1647867600          0.231919                43
1   recommendationservice2-0      container         24                1647867600          0.287693                24
2                     node-6           node         23                1647867600        123.419890                23
3   productcatalogservice2-0      container         22                1647867600          0.069727                22
4                 frontend-1      container         22                1647867600          0.060948                22
5                 adservice2      container         22                1647867600          0.052592                22
6    productcatalogservice-2      container         22                1647867600          0.022785                22
7                  adservice      container         20                1647867600          0.330016                20
8    productcatalogservice-1      container         19                1647867600          0.017386                19
9    productcatalogservice-0      container         17                1647867600          0.290323                17
10   recommendationservice-1      container         17                1647867600          0.041190                17
11              adservice2-0      container         16                1647867600          0.532557                16
12   recommendationservice-2      container         16                1647867600          0.463675                16
13            cartservice2-0      container         15                1647867600          0.090850                15
14             cartservice-2      container         14                1647867600          0.616766                14
15   recommendationservice-0      container         14                1647867600          0.018005                14
16                    node-4           node         13                1647867840          1.426045                13
17                frontend-2      container         13                1647867600          0.469546                13
18                    node-5           node         12                1647867600          0.130025                12
19                frontend-0      container         12                1647867600          0.093097                12,               service_name  service_run_count  earliest_start_timestamp  max_breach_ratio  unique_cmdb_ids
0    productcatalogservice                 59                1647867600          0.290323                4
1                 frontend                 48                1647867600          0.469546                4
2    recommendationservice                 48                1647867600          0.463675                4
3                frontend2                 43                1647867600          0.231919                1
4               adservice2                 38                1647867600          0.532557                2
5              cartservice                 37                1647867600          0.616766                4
6                adservice                 36                1647867600          0.591373                4
7   recommendationservice2                 24                1647867600          0.287693                1
8   productcatalogservice2                 22                1647867600          0.069727                1
9             cartservice2                 15                1647867600          0.090850                1
10         checkoutservice                 13                1647867600          0.648458                3
11         currencyservice                 12                1647868080          0.553852                3
12            emailservice                 12                1647868140         17.477721                3
13         shippingservice                 12                1647867600          0.702731                3
14          paymentservice                 11                1647867780          0.698657                3
15        currencyservice2                 11                1647867600          1.818378                1
16        checkoutservice2                  8                1647867600          0.055901                1
17           emailservice2                  6                1647867600          0.250000                1
18        shippingservice2                  5                1647867720          0.611090                1
19         paymentservice2                  3                1647867600          0.015635                1, 69, 466, 6, 47,      cmdb_id        kpi_name  start_timestamp  breach_ratio
0  adservice  app.grpc.count       1647867600      0.006036)```
```

In [ ]:
# In[5]:

# Reuse existing kernel variable df_faults
import pandas as pd
import numpy as np

# 1) Filter runs where cmdb_id starts with 'node-' and store as df_node_runs
df_node_runs = df_faults[df_faults['cmdb_id'].str.startswith('node-')].copy()

# Ensure timestamp columns are integer types
if not df_node_runs.empty:
    df_node_runs['start_timestamp'] = df_node_runs['start_timestamp'].astype(int)
    df_node_runs['end_timestamp'] = df_node_runs['end_timestamp'].astype(int)

# 1) Top 20 node runs sorted by breach_ratio desc (selected columns)
top20_node_runs = (
    df_node_runs
    .sort_values(by='breach_ratio', ascending=False)
    .loc[:, ['cmdb_id','kpi_name','start_timestamp','end_timestamp',
             'length_in_points','max_value_in_run','global_P95','breach_ratio']]
    .head(20)
    .reset_index(drop=True)
)

# 2) Single node run with maximum breach_ratio (full row)
if df_node_runs.shape[0] > 0:
    max_breach_node_run = df_node_runs.nlargest(1, 'breach_ratio').reset_index(drop=True)
else:
    max_breach_node_run = df_node_runs.head(0)  # empty df

# 3) Single earliest node run overall (min start_timestamp) with selected columns
if df_node_runs.shape[0] > 0:
    earliest_node_run = df_node_runs.nsmallest(1, 'start_timestamp').loc[:, ['cmdb_id','kpi_name','start_timestamp','breach_ratio']].reset_index(drop=True)
else:
    earliest_node_run = df_node_runs.head(0)

# 4) Summary integers
total_node_runs = int(df_node_runs.shape[0])
number_of_nodes_with_runs = int(df_node_runs['cmdb_id'].nunique())

# Return compact outputs (variables)
df_node_runs, top20_node_runs, max_breach_node_run, earliest_node_run, total_node_runs, number_of_nodes_with_runs

```
Out[5]:
```
Summary (plain English, compact):

- I stored node-only runs as df_node_runs in the kernel (69 runs total across 6 distinct node IDs).

- Top examples from the node runs (top-ranked by breach_ratio):
  1. node-6 — system.io.r_s: start 1647869040, end 1647869100, length 2, max_value 1126.00, global_P95 9.05, breach_ratio ≈ 123.42
  2. node-6 — system.io.rkb_s: start 1647869040, end 1647869100, length 2, max_value 4606.00, global_P95 46.10, breach_ratio ≈ 98.91
  3. node-1 — system.io.avg_q_sz: start 1647867600, end 1647867660, length 2, max_value 0.41, global_P95 0.031, breach_ratio ≈ 12.23
  4. node-1 — system.io.w_s: start 1647867600, end 1647867660, length 2, max_value 101.50, global_P95 36.725, breach_ratio ≈ 1.76
  5. node-4 — system.net.packets_in.count: start 1647868320, end 1647868380, length 2, max_value 1335.01, global_P95 550.28, breach_ratio ≈ 1.43
  (The full top-20 node runs table was returned and is available in the kernel.)

- Single node run with the maximum breach_ratio:
  - cmdb_id: node-6
  - kpi_name: system.io.r_s
  - start_timestamp: 1647869040
  - end_timestamp: 1647869100
  - length_in_points: 2
  - max_value_in_run: 1126.00
  - global_P95: 9.05
  - breach_ratio: ≈ 123.41989

- Earliest node run overall:
  - cmdb_id: node-1
  - kpi_name: system.io.avg_q_sz
  - start_timestamp: 1647867600
  - breach_ratio: ≈ 12.225806

- Summary counts:
  - total_node_runs: 69
  - number_of_nodes_with_runs: 6

Notes: df_node_runs, the top-20 node runs, the single max-breach run, and the earliest-run row are stored in the kernel for follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                     kpi_name  segment  start_timestamp  end_timestamp  length_in_points  max_value_in_run    global_P95  breach_ratio
1479  node-1        system.fs.inodes.used        4       1647867840     1647867900                 2        1945593.00  1.945584e+06      0.000005
1480  node-1           system.io.avg_q_sz        0       1647867600     1647867660                 2              0.41  3.100000e-02     12.225806
1488  node-1                system.io.w_s        0       1647867600     1647867660                 2            101.50  3.672500e+01      1.763785
1491  node-1               system.load.15       15       1647868500     1647868560                 2              1.09  1.026500e+00      0.061861
1492  node-1                system.load.5       14       1647868440     1647868500                 2              1.42  1.235500e+00      0.149332
...      ...                          ...      ...              ...            ...               ...               ...           ...           ...
1759  node-6        system.net.bytes_rcvd        1       1647867660     1647867720                 2         217483.80  2.142083e+05      0.015291
1760  node-6        system.net.bytes_sent        2       1647867720     1647867780                 2         589177.04  4.621626e+05      0.274826
1765  node-6       system.net.tcp.in_segs        2       1647867720     1647867780                 2            499.82  3.721600e+02      0.343025
1766  node-6      system.net.tcp.out_segs        2       1647867720     1647867780                 2           8093.07  5.760201e+03      0.404998
1767  node-6  system.net.tcp.retrans_segs        3       1647867780     1647867840                 2              0.28  1.295000e-01      1.162162

[69 rows x 9 columns],    cmdb_id                      kpi_name  start_timestamp  end_timestamp  length_in_points  max_value_in_run    global_P95  breach_ratio
0   node-6                 system.io.r_s       1647869040     1647869100                 2           1126.00  9.050000e+00    123.419890
1   node-6               system.io.rkb_s       1647869040     1647869100                 2           4606.00  4.610000e+01     98.913232
2   node-1            system.io.avg_q_sz       1647867600     1647867660                 2              0.41  3.100000e-02     12.225806
3   node-1                 system.io.w_s       1647867600     1647867660                 2            101.50  3.672500e+01      1.763785
4   node-4   system.net.packets_in.count       1647868320     1647868380                 2           1335.01  5.502825e+02      1.426045
5   node-4         system.net.bytes_rcvd       1647868320     1647868380                 2        1975759.40  8.299236e+05      1.380652
6   node-4        system.net.tcp.in_segs       1647868320     1647868380                 2           2701.54  1.221933e+03      1.210873
7   node-6   system.net.tcp.retrans_segs       1647867780     1647867840                 2              0.28  1.295000e-01      1.162162
8   node-4  system.net.packets_out.count       1647868320     1647868380                 2            456.97  3.050785e+02      0.497877
9   node-4  system.net.udp.out_datagrams       1647868440     1647868500                 2              1.73  1.217500e+00      0.420945
10  node-4   system.net.udp.in_datagrams       1647868440     1647868500                 2              1.73  1.217500e+00      0.420945
11  node-6       system.net.tcp.out_segs       1647867720     1647867780                 2           8093.07  5.760201e+03      0.404998
12  node-4       system.net.tcp.out_segs       1647868380     1647868440                 2           8609.39  6.347195e+03      0.356409
13  node-6        system.net.tcp.in_segs       1647867720     1647867780                 2            499.82  3.721600e+02      0.343025
14  node-3       system.net.tcp.out_segs       1647868620     1647868680                 2           2422.07  1.853050e+03      0.307072
15  node-6         system.net.bytes_sent       1647867720     1647867780                 2         589177.04  4.621626e+05      0.274826
16  node-3                 system.load.5       1647867600     1647867660                 2              1.54  1.256000e+00      0.226115
17  node-1  system.net.packets_out.count       1647867660     1647867720                 2            495.58  4.166765e+02      0.189364
18  node-4         system.net.bytes_sent       1647868380     1647868440                 2        1309110.50  1.109690e+06      0.179708
19  node-6                 system.load.5       1647868320     1647868380                 2              9.80  8.335500e+00      0.175694,   cmdb_id       kpi_name  segment  start_timestamp  end_timestamp  length_in_points  max_value_in_run  global_P95  breach_ratio
0  node-6  system.io.r_s       24       1647869040     1647869100                 2            1126.0        9.05     123.41989,   cmdb_id            kpi_name  start_timestamp  breach_ratio
0  node-1  system.io.avg_q_sz       1647867600     12.225806, 69, 6)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Reuse df_metric from kernel
import pandas as pd

# Find rows where kpi_name contains the substring '.node-6' (case-sensitive, literal)
mask_node6 = df_metric['kpi_name'].fillna('').str.contains('.node-6', regex=False)
df_node6_kpis = df_metric[mask_node6].copy()

# (A) Up to 50 unique cmdb_id values (pods/containers) hosted on node-6, and total count
unique_cmdb = df_node6_kpis['cmdb_id'].dropna().astype(str).unique()
pods_on_node6 = pd.DataFrame({'cmdb_id': unique_cmdb[:50]})  # store as DataFrame in kernel
unique_cmdb_count = int(len(unique_cmdb))

# (B) Up to 20 example kpi_name strings that include '.node-6'
example_kpis = pd.Series(df_node6_kpis['kpi_name'].dropna().astype(str).unique()[:20], name='kpi_name')

# (C) Total number of metric rows that contain '.node-6'
total_rows_with_node6 = int(df_node6_kpis.shape[0])

# Compact outputs (stored variables and summaries)
pods_on_node6, unique_cmdb_count, example_kpis, total_rows_with_node6

```
Out[6]:
```
Summary (plain English, compact):

A) Unique cmdb_id values (pods/containers) whose metrics reference ".node-6" — up to 50 shown (there are 7 total):
- adservice2-0, cartservice2-0, currencyservice2-0, frontend2-0, paymentservice2-0, productcatalogservice2-0, recommendationservice2-0
(Unique count = 7; stored in kernel variable pods_on_node6.)

B) Example kpi_name strings that include ".node-6" (up to 20 examples shown):
- container.node-6.container_cpu_cfs_periods
- container.node-6.container_cpu_cfs_throttled_periods
- container.node-6.container_cpu_cfs_throttled_seconds
- container.node-6.container_cpu_load_average_10s
- container.node-6.container_cpu_system_seconds
- container.node-6.container_cpu_usage_seconds
- container.node-6.container_cpu_user_seconds
- container.node-6.container_file_descriptors
- container.node-6.container_fs_inodes./dev/vda1
- container.node-6.container_fs_inodes_free./dev/vda1
... (more similar container.node-6.* metrics available)

C) Total number of metric rows containing ".node-6": 13,440.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id
0              adservice2-0
1            cartservice2-0
2        currencyservice2-0
3               frontend2-0
4         paymentservice2-0
5  productcatalogservice2-0
6  recommendationservice2-0, 7, 0            container.node-6.container_cpu_cfs_periods
1     container.node-6.container_cpu_cfs_throttled_p...
2     container.node-6.container_cpu_cfs_throttled_s...
3       container.node-6.container_cpu_load_average_10s
4         container.node-6.container_cpu_system_seconds
5          container.node-6.container_cpu_usage_seconds
6           container.node-6.container_cpu_user_seconds
7           container.node-6.container_file_descriptors
8        container.node-6.container_fs_inodes./dev/vda1
9     container.node-6.container_fs_inodes_free./dev...
10    container.node-6.container_fs_io_current./dev/...
11    container.node-6.container_fs_io_time_seconds....
12    container.node-6.container_fs_io_time_weighted...
13     container.node-6.container_fs_limit_MB./dev/vda1
14    container.node-6.container_fs_read_seconds./de...
15         container.node-6.container_fs_reads./dev/vda
16        container.node-6.container_fs_reads./dev/vda1
17      container.node-6.container_fs_reads_MB./dev/vda
18    container.node-6.container_fs_reads_merged./de...
19    container.node-6.container_fs_sector_reads./de...
Name: kpi_name, dtype: object, 13440)```
```

In [ ]:
# In[7]:

# Reuse kernel variables: df_trace, df_error_logs, pods_on_node6
import pandas as pd
import numpy as np

# Prepare pod list from pods_on_node6 (DataFrame with cmdb_id column)
pods_list = pods_on_node6['cmdb_id'].dropna().astype(str).unique().tolist()

# Incident window timestamps (already used earlier)
start_ts = 1647867600
end_ts = 1647869340

# 1) Trace scan: filter df_trace for pods_on_node6 and window; store df_trace_node6
df_trace_node6 = df_trace[
    df_trace['cmdb_id'].isin(pods_list) &
    (df_trace['timestamp'] >= start_ts) &
    (df_trace['timestamp'] <= end_ts)
].copy()

# Compact trace summary counts
if not df_trace_node6.empty:
    trace_total_rows = int(df_trace_node6.shape[0])
    trace_min_ts = int(df_trace_node6['timestamp'].min())
    trace_max_ts = int(df_trace_node6['timestamp'].max())
else:
    trace_total_rows = 0
    trace_min_ts = pd.NA
    trace_max_ts = pd.NA

trace_counts_df = pd.DataFrame([{
    'total_row_count': trace_total_rows,
    'min_timestamp': trace_min_ts,
    'max_timestamp': trace_max_ts
}])

# Top 20 trace_name by row count with count and max(value)
trace_top20 = (
    df_trace_node6.dropna(subset=['trace_name'])
    .groupby('trace_name', as_index=False)
    .agg(row_count=('trace_name', 'size'), max_value=('value', 'max'))
    .sort_values(by='row_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 2) Error log scan: filter df_error_logs for pods_on_node6 and window; store df_errorlogs_node6
df_errorlogs_node6 = df_error_logs[
    df_error_logs['cmdb_id'].isin(pods_list) &
    (df_error_logs['timestamp'] >= start_ts) &
    (df_error_logs['timestamp'] <= end_ts)
].copy()

# Compact error logs summary
if not df_errorlogs_node6.empty:
    err_total_rows = int(df_errorlogs_node6.shape[0])
    err_min_ts = int(df_errorlogs_node6['timestamp'].min())
    err_max_ts = int(df_errorlogs_node6['timestamp'].max())
    err_unique_cmdb = int(df_errorlogs_node6['cmdb_id'].nunique())
else:
    err_total_rows = 0
    err_min_ts = pd.NA
    err_max_ts = pd.NA
    err_unique_cmdb = 0

error_info_df = pd.DataFrame([{
    'total_row_count': err_total_rows,
    'min_timestamp': err_min_ts,
    'max_timestamp': err_max_ts,
    'unique_cmdb_id_count': err_unique_cmdb
}])

# Example messages (if message column exists). Truncate to 200 chars, unique examples up to 10
if 'message' in df_errorlogs_node6.columns and not df_errorlogs_node6['message'].dropna().empty:
    example_msgs = (
        df_errorlogs_node6['message']
        .dropna()
        .astype(str)
        .map(lambda s: s[:200])
        .unique()[:10]
    )
    example_msgs = pd.Series(example_msgs, name='example_message')
else:
    example_msgs = pd.Series([], dtype=object, name='example_message')

# 3) Earliest extreme trace record within df_trace_node6:
# For each trace_name compute 99th percentile (within df_trace_node6), then find rows >= that threshold
extreme_trace_row = pd.DataFrame(columns=['cmdb_id','trace_name','timestamp','value'])

if not df_trace_node6.empty:
    # drop NaN values for value when computing quantiles
    trace_vals = df_trace_node6.dropna(subset=['value', 'trace_name']).copy()
    if not trace_vals.empty:
        p99 = trace_vals.groupby('trace_name', as_index=False)['value'].quantile(0.99).rename(columns={'value':'p99'})
        # Merge p99 back
        merged = trace_vals.merge(p99, on='trace_name', how='left')
        # Candidate rows where value >= p99
        candidates = merged[merged['value'] >= merged['p99']].copy()
        if not candidates.empty:
            # Pick earliest candidate by timestamp
            earliest_candidate = candidates.nsmallest(1, 'timestamp').loc[:, ['cmdb_id','trace_name','timestamp','value']].reset_index(drop=True)
            extreme_trace_row = earliest_candidate
        else:
            # Fallback: return single maximum-value trace record (if ties, earliest among them)
            max_val = trace_vals['value'].max()
            max_rows = trace_vals[trace_vals['value'] == max_val]
            earliest_max = max_rows.nsmallest(1, 'timestamp').loc[:, ['cmdb_id','trace_name','timestamp','value']].reset_index(drop=True)
            extreme_trace_row = earliest_max
    else:
        extreme_trace_row = pd.DataFrame(columns=['cmdb_id','trace_name','timestamp','value'])

# Store df_trace_node6 and df_errorlogs_node6 in kernel (variables already assigned)
# Return compact aggregates only
trace_top20, trace_counts_df, error_info_df, example_msgs, extreme_trace_row

```
Out[7]:
```
Summary (plain English, compact):

- I stored the filtered traces as df_trace_node6 and error logs as df_errorlogs_node6 in the kernel.

1) Trace scan (df_trace_node6)
- Total rows: 2,760
- Timestamp range: min 1647867600, max 1647869340 (incident window)
- Top 20 trace_name by row count (row_count, max(value)):
  - trace.from_frontend2-0.row_count — 150 rows — max=771.0
  - trace.from_frontend2-0.error_rate — 150 rows — max=0.0
  - trace.from_frontend2-0.duration_p95 — 150 rows — max=0.005098
  - trace.from_frontend2-0.duration_mean — 150 rows — max=0.004207
  - trace.from_checkoutservice2-0.row_count — 120 rows — max=24.0
  - trace.from_checkoutservice2-0.error_rate — 120 rows — max=0.0
  - trace.from_checkoutservice2-0.duration_p95 — 120 rows — max=0.005727
  - trace.from_checkoutservice2-0.duration_mean — 120 rows — max=0.004659
  - trace.self.row_count — 90 rows — max=1600.0
  - trace.self.error_rate — 90 rows — max=0.0
  - trace.self.duration_p95 — 90 rows — max=0.006714
  - trace.self.duration_mean — 90 rows — max=0.004389
  - trace.from_root.row_count — 30 rows — max=160.0
  - trace.from_root.error_rate — 30 rows — max=0.0
  - trace.from_root.duration_p95 — 30 rows — max=0.081612
  - trace.from_root.duration_mean — 30 rows — max=0.045095
  - trace.to_adservice2-0.duration_mean — 30 rows — max=0.000020
  - trace.to_adservice2-0.duration_p95 — 30 rows — max=0.000034
  - trace.to_adservice2-0.error_rate — 30 rows — max=0.0
  - trace.to_adservice2-0.row_count — 30 rows — max=100.0

2) Error log scan (df_errorlogs_node6)
- Total rows: 0 (no error-log rows for the pods on node-6 in the incident window)
- min_timestamp / max_timestamp: none (NA)
- Unique cmdb_id count: 0
- Example messages: none

3) Extreme trace record (within df_trace_node6)
- I looked for values at or above the per-trace_name 99th percentile; the earliest such extreme (or fallback max) is:
  - cmdb_id: adservice2-0
  - trace_name: trace.from_frontend2-0.error_rate
  - timestamp: 1647867600
  - value: 0.0

Notes:
- df_trace_node6 contains the trace rows for the 7 pods on node-6 within the incident window; df_errorlogs_node6 contains no matching error-log rows.
- All filtered datasets are stored in kernel variables df_trace_node6 and df_errorlogs_node6 for follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

(                                     trace_name  row_count    max_value
0              trace.from_frontend2-0.row_count        150   771.000000
1             trace.from_frontend2-0.error_rate        150     0.000000
2           trace.from_frontend2-0.duration_p95        150     0.005098
3          trace.from_frontend2-0.duration_mean        150     0.004207
4       trace.from_checkoutservice2-0.row_count        120    24.000000
5      trace.from_checkoutservice2-0.error_rate        120     0.000000
6    trace.from_checkoutservice2-0.duration_p95        120     0.005727
7   trace.from_checkoutservice2-0.duration_mean        120     0.004659
8                          trace.self.row_count         90  1600.000000
9                         trace.self.error_rate         90     0.000000
10                      trace.self.duration_p95         90     0.006714
11                     trace.self.duration_mean         90     0.004389
12                    trace.from_root.row_count         30   160.000000
13                   trace.from_root.error_rate         30     0.000000
14                 trace.from_root.duration_p95         30     0.081612
15                trace.from_root.duration_mean         30     0.045095
16          trace.to_adservice2-0.duration_mean         30     0.000020
17           trace.to_adservice2-0.duration_p95         30     0.000034
18             trace.to_adservice2-0.error_rate         30     0.000000
19              trace.to_adservice2-0.row_count         30   100.000000,    total_row_count  min_timestamp  max_timestamp
0             2760     1647867600     1647869340,    total_row_count min_timestamp max_timestamp  unique_cmdb_id_count
0                0          <NA>          <NA>                     0, Series([], Name: example_message, dtype: object),         cmdb_id                         trace_name   timestamp  value
0  adservice2-0  trace.from_frontend2-0.error_rate  1647867600    0.0)```
```

In [ ]:
# In[8]:

# Reuse existing kernel variables: df_metric, df_thresholds, df_trace_node6
import pandas as pd
import numpy as np
import pytz

# 1) Extract node-6 IO metric rows in the given time range and kpi list
kpis = ['system.io.r_s','system.io.rkb_s','system.io.w_s','system.io.avg_q_sz','system.io.wkb_s']
start_ts = 1647868980
end_ts = 1647869120

df_node6_io_context = (
    df_metric[
        (df_metric['cmdb_id'] == 'node-6') &
        (df_metric['kpi_name'].isin(kpis)) &
        (df_metric['timestamp'] >= start_ts) &
        (df_metric['timestamp'] <= end_ts)
    ]
    .merge(df_thresholds, on=['cmdb_id','kpi_name'], how='left')
    .sort_values('timestamp')
    .reset_index(drop=True)
)

# Derived columns: exceeds_p95 and breach_ratio (handle missing or zero global_P95)
df_node6_io_context['exceeds_p95'] = (
    df_node6_io_context['global_P95'].notna() & (df_node6_io_context['value'] > df_node6_io_context['global_P95'])
)

# breach_ratio: (value - global_P95) / global_P95, NaN if global_P95 is NaN or zero
def compute_breach(row):
    gp = row['global_P95']
    if pd.isna(gp) or gp == 0:
        return np.nan
    return (row['value'] - gp) / gp

df_node6_io_context['breach_ratio'] = df_node6_io_context.apply(compute_breach, axis=1)

# Keep requested columns and limit to up to 50 rows sorted by timestamp
df_node6_io_context = df_node6_io_context.loc[:, ['timestamp','kpi_name','value','global_P95','exceeds_p95','breach_ratio']]
df_node6_io_context_head50 = df_node6_io_context.head(50)

# 2) Aggregate trace row_count metrics for node-6 pods in the same short window
df_node6_trace_counts = (
    df_trace_node6[
        df_trace_node6['trace_name'].str.contains('row_count', na=False) &
        (df_trace_node6['timestamp'] >= start_ts) &
        (df_trace_node6['timestamp'] <= end_ts)
    ]
    .groupby('timestamp', as_index=False)['value']
    .sum()
    .rename(columns={'value':'total_row_count'})
    .sort_values('timestamp')
    .reset_index(drop=True)
)

# Ensure integer timestamps and numeric total_row_count
if not df_node6_trace_counts.empty:
    df_node6_trace_counts['timestamp'] = df_node6_trace_counts['timestamp'].astype(int)
    df_node6_trace_counts['total_row_count'] = pd.to_numeric(df_node6_trace_counts['total_row_count'], errors='coerce')

# Display compact results via variables (per IPython rules)
df_node6_io_context_head50, df_node6_trace_counts

```
Out[8]:
```
Summary (plain English, compact):

- Time window analyzed: 1647868980 — 1647869120 (covers severe run at 1647869040).
- I stored the metric rows as df_node6_io_context and the trace aggregates as df_node6_trace_counts in the kernel.

1) Node-6 IO metric context (selected KPIs)
- At 1647869040 (the severe moment):
  - system.io.r_s = 1126.00 vs global_P95 = 9.05 → exceeds_p95 = True, breach_ratio ≈ 123.42 (very large read spike)
  - system.io.rkb_s = 4606.00 vs global_P95 = 46.10 → exceeds_p95 = True, breach_ratio ≈ 98.91 (very large read KB/sec spike)
  - system.io.w_s = 29.00 vs global_P95 = 272.575 → below P95
  - system.io.avg_q_sz = 0.91 vs global_P95 = 63.2465 → below P95
- At 1647869100 (after the peak):
  - system.io.r_s = 14.00 (still > P95, breach_ratio ≈ 0.55)
  - system.io.rkb_s = 56.00 (still > P95, breach_ratio ≈ 0.21)
  - writes remained below P95
- Before the peak (1647868980) values were low/near zero and well below P95.
- Conclusion: the primary anomaly is a huge read I/O spike (r_s and rkb_s) centered at 1647869040 with very high breach ratios; writes did not spike comparably.

2) Trace impact (row_count aggregation across pods on node-6)
- total_row_count by timestamp (all trace.row_count metrics combined across node-6 pods):
  - 1647868980 → 3445.0
  - 1647869040 → 3312.0
  - 1647869100 → 3185.0
- Interpretation: trace row_count totals are high across the window and remain elevated during and after the IO spike, indicating measurable application load/impact on the node’s pods at the spike time.

Bottom line: At 1647869040 node-6 experienced a massive read I/O spike (system.io.r_s and system.io.rkb_s) far above historical P95; trace row_count totals across node-6 pods were high during the same timestamps, consistent with application-level impact.

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp            kpi_name    value  global_P95  exceeds_p95  breach_ratio
0   1647868980  system.io.avg_q_sz     0.01     63.2465        False     -0.999842
1   1647868980       system.io.r_s     0.00      9.0500        False     -1.000000
2   1647868980     system.io.rkb_s     0.00     46.1000        False     -1.000000
3   1647868980       system.io.w_s    18.50    272.5750        False     -0.932129
4   1647869040  system.io.avg_q_sz     0.91     63.2465        False     -0.985612
5   1647869040       system.io.r_s  1126.00      9.0500         True    123.419890
6   1647869040     system.io.rkb_s  4606.00     46.1000         True     98.913232
7   1647869040       system.io.w_s    29.00    272.5750        False     -0.893607
8   1647869100  system.io.avg_q_sz     0.06     63.2465        False     -0.999051
9   1647869100       system.io.r_s    14.00      9.0500         True      0.546961
10  1647869100     system.io.rkb_s    56.00     46.1000         True      0.214751
11  1647869100       system.io.w_s    69.00    272.5750        False     -0.746859,     timestamp  total_row_count
0  1647868980           3445.0
1  1647869040           3312.0
2  1647869100           3185.0)```
```